<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/projects/06-Function-Calling-%26-Data_Extraction/05_Applications_of_FunctionCalling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q groq

from groq import Groq
from google.colab import userdata
import inspect, json
import requests

GROQ_API_KEY = userdata.get('groq_key')
client = Groq(api_key=GROQ_API_KEY)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.8 MB/s eta 0:00:00


In [4]:
def get_completion(prompt, model="llama-3.3-70b-versatile"):
  messages = [{'role':'system','content': "You are being used as a tool calling agent, just respond with the function call ONLY"},
              {"role": "user", "content": prompt}]
  response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0,)
  return response.choices[0].message.content

In [8]:
def get_completion_no_func(prompt, model="llama-3.3-70b-versatile"):
  messages = [{"role": "user", "content": prompt}]
  response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0,)
  return response.choices[0].message.content

In [5]:
def get_completion_from_messages(messages, tools,
                                 model="llama-3.3-70b-versatile",
                                 temperature=0, ):
    response = client.chat.completions.create(
        model=model,
        tools=tools,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.tool_calls[0]

In [6]:
def build_prompt(function_list, user_query):
  f_prompt = ""
  for function in function_list:
    signature = inspect.signature(function)
    docstring = function.__doc__
    prompt= f''' Function:
def {function.__name__}{signature}
  """
  {docstring.strip()}
  """
     '''
    f_prompt += prompt
  f_prompt+= f"User Query: {user_query}<human_end>"
  return f_prompt

In [9]:
question = "Hey, can you tell me more about this R1 thing that was announced by Rabbit? "

no_function_calling_prompt = \
f"""
<s> [INST] {question} [/INST]
"""
get_completion_no_func(no_function_calling_prompt)

"You're referring to the R1 announcement by Rabbit. Unfortunately, I don't have have specific information on what the R1 is, as my knowledge stopped in 2023 and I'm not aware of any recent announcements. However, I can suggest some possible ways to find more information about it.\n\nRabbit is likely a company or organization that has made an announcement about a product, service, or project called R1. To learn more, you can try checking the company's official website, social media, or press releases for any updates or announcements related to R1.\n\nAdditionally, you can try searching online for news articles or blogs that may have covered the announcement. This can give you a better understanding of what R1 is and what it's intended for.\n\nIf you have any more specific information about the R1 or Rabbit, such as the industry or context in which it was announced, I may be able to provide more general information or point you in the direction of resources that could be helpful."

## Providing Up To Date Information

In [ ]:
def do_web_search(full_user_prompt : str, num_results : int = 5):
    API_URL = f'https://api.tavily.com/search'
    payload = \
    {
      "api_key": userdata.get('tavily_key'),
      "query": full_user_prompt,
      "search_depth": "basic",
      "include_answer": False,
      "include_images": False,
      "include_raw_content": False,
      "max_results": num_results,
      "include_domains": [],
      "exclude_domains": []
    }
    import requests
    response = requests.post(API_URL, json=payload)
    response = response.json()
    all_results = "\n\n".join(item["content"] for item in response["results"])
    return all_results